In [ ]:
import pandas as pd
import io

full_data = pd.read_csv('https://dpl6hyzg28thp.cloudfront.net/media/full_economic_data.csv', parse_dates=['DATE'], index_col='DATE')

full_data = full_data.asfreq('QS')
full_data = full_data.dropna()

print(len(full_data), "quarters from", full_data.index[0].date(), "to", full_data.index[-1].date())
full_data

In [ ]:
from sklearn.preprocessing import StandardScaler

y = full_data['GDPC1']
exog = full_data.drop(columns='GDPC1')

scaler = StandardScaler()
exog_scaled = pd.DataFrame(scaler.fit_transform(exog), columns=exog.columns, index=exog.index)

In [ ]:
split_idx = int(len(full_data) * 0.8)

y_train = y[:split_idx]
y_test = y[split_idx:]

exog_train = exog_scaled[:split_idx]
exog_test = exog_scaled[split_idx:]

print("train:", len(y_train), "test:", len(y_test))

In [ ]:
# check how much the minimum wage actually changes in our data
mw = full_data['FEDMINNFRWG']
changes = (mw.diff().fillna(0) != 0).sum()
print("minimum wage changes", changes, "times, values:", sorted(mw.unique()))
print("last change:", mw.index[(mw.diff().fillna(0) != 0)][-1].date())

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

exog_vars_ctl = ['CIVPART', 'CPIAUCSL', 'UMCSENT', 'PAYEMS', 'PCE']
exog_vars_exp = exog_vars_ctl + ['FEDMINNFRWG']

def init_model(exog_vars):
  return SARIMAX(
      y_train,
      exog=exog_train[exog_vars],
      order=(1, 1, 1),
      enforce_stationarity=False,
      enforce_invertibility=False
  )

model_ctl = init_model(exog_vars_ctl)
model_exp = init_model(exog_vars_exp)

results_ctl = model_ctl.fit(method='powell', maxiter=1000)
results_exp = model_exp.fit(method='powell', maxiter=1000)

print("AIC without min wage:", round(results_ctl.aic, 1))
print("AIC with min wage:", round(results_exp.aic, 1))

In [ ]:
from scipy import stats

# z-test on the minimum wage coefficient
coef = results_exp.params['FEDMINNFRWG']
z = results_exp.zvalues['FEDMINNFRWG']
p = results_exp.pvalues['FEDMINNFRWG']
print(f"min wage coefficient = {coef:.2f}, z = {z:.2f}, p = {p:.2f}")

# likelihood ratio test between the two SARIMAX models
LR_stat = 2 * (results_exp.llf - results_ctl.llf)
p_value = stats.chi2.sf(LR_stat, 1)
print(f"Likelihood Ratio Statistic: {LR_stat:.2f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fc_ctl = results_ctl.get_forecast(len(y_test), exog=exog_test[exog_vars_ctl])
fc_exp = results_exp.get_forecast(len(y_test), exog=exog_test[exog_vars_exp])
forecast_ctl = fc_ctl.predicted_mean
forecast_exp = fc_exp.predicted_mean
ci = fc_ctl.conf_int()

plt.figure(figsize=(10, 5))
plt.plot(y_test.index, y_test / 1000, color='#1f3b6e', lw=2.4, marker='o', ms=4, label='Actual GDP')
plt.plot(y_test.index, forecast_ctl / 1000, color='#c0392b', lw=2, ls='--', marker='s', ms=3.5, label='Without minimum wage')
plt.plot(y_test.index, forecast_exp / 1000, color='#117733', lw=2, ls=':', marker='^', ms=4, label='With minimum wage')
plt.fill_between(y_test.index, ci.iloc[:, 0] / 1000, ci.iloc[:, 1] / 1000, color='#c0392b', alpha=0.12, label='95% CI (without min. wage)')
plt.xlabel('Year')
plt.ylabel('Real GDP (trillions of chained 2017 US$)')
plt.gca().xaxis.set_major_locator(mdates.YearLocator(2))
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('fig1_sarimax.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
import statsmodels.formula.api as smf

data_for_lm = pd.DataFrame({"previous_GDP": y_train.values[:-1], "current_GDP": y_train.values[1:]}, index=y_train.index[1:])
for exog_var in exog_vars_exp:
  data_for_lm[exog_var] = exog_train[exog_var].values[1:]

depvar = 'current_GDP'
mod1 = smf.ols(formula=depvar + ' ~ previous_GDP + ' + " + ".join(exog_vars_ctl), data=data_for_lm)
mod1 = mod1.fit()

mod2 = smf.ols(formula=depvar + ' ~ previous_GDP + ' + " + ".join(exog_vars_ctl) + " + FEDMINNFRWG", data=data_for_lm)
mod2 = mod2.fit()

print("linear AIC without min wage:", round(mod1.aic, 1))
print("linear AIC with min wage:", round(mod2.aic, 1))

# t-test on the minimum wage coefficient
print(f"min wage coefficient = {mod2.params['FEDMINNFRWG']:.2f}, t = {mod2.tvalues['FEDMINNFRWG']:.2f}, p = {mod2.pvalues['FEDMINNFRWG']:.2f}")

# likelihood ratio test between the two linear models
LR_stat = 2 * (mod2.llf - mod1.llf)
p_value = stats.chi2.sf(LR_stat, 1)
print(f"Likelihood Ratio Statistic: {LR_stat:.2f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
# multi-step forecasts for the linear models (feed each prediction back in as previous_GDP,
# same idea as the SARIMAX forecast)
def linear_forecast(mod, exog_vars):
  prev = y_train.values[-1]
  preds = []
  for t in range(len(y_test)):
    row = {'previous_GDP': prev}
    for v in exog_vars:
      row[v] = exog_test[v].values[t]
    pred = float(mod.predict(pd.DataFrame([row])).iloc[0])
    preds.append(pred)
    prev = pred
  return pd.Series(preds, index=y_test.index)

lin_forecast_ctl = linear_forecast(mod1, exog_vars_ctl)
lin_forecast_exp = linear_forecast(mod2, exog_vars_exp)

plt.figure(figsize=(10, 5))
plt.plot(y_test.index, y_test / 1000, color='#1f3b6e', lw=2.4, marker='o', ms=4, label='Actual GDP')
plt.plot(y_test.index, lin_forecast_ctl / 1000, color='#c0392b', lw=2, ls='--', marker='s', ms=3.5, label='Without minimum wage')
plt.plot(y_test.index, lin_forecast_exp / 1000, color='#117733', lw=2, ls=':', marker='^', ms=4, label='With minimum wage')
plt.xlabel('Year')
plt.ylabel('Real GDP (trillions of chained 2017 US$)')
plt.gca().xaxis.set_major_locator(mdates.YearLocator(2))
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('fig2_linear.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def report(name, forecast):
  mse = mean_squared_error(y_test, forecast)
  rmse = np.sqrt(mse)
  mape = np.mean(np.abs((y_test - forecast) / y_test)) * 100
  print(f"{name}: MSE = {mse:,.0f}, RMSE = {rmse:,.0f}, MAPE = {mape:.2f}%")

report("Linear without min wage ", lin_forecast_ctl)
report("Linear with min wage    ", lin_forecast_exp)
report("SARIMAX without min wage", forecast_ctl)
report("SARIMAX with min wage   ", forecast_exp)

In [ ]:
import itertools

optional_vars = [var for var in exog.columns if var != 'FEDMINNFRWG']

search_results = []
best_aic = float('inf')
best_subset = None

print(f"\U0001F50D Searching over all {2**len(optional_vars)} control combinations, with and without minimum wage...")

count = 0
for r in range(len(optional_vars) + 1):
    for subset in itertools.combinations(optional_vars, r):
        for with_mw in (False, True):
            try:
                full_subset = (['FEDMINNFRWG'] if with_mw else []) + list(subset)
                if full_subset:
                    model = SARIMAX(
                        y_train,
                        exog=exog_train[full_subset],
                        order=(1, 1, 1),
                        enforce_stationarity=False,
                        enforce_invertibility=False
                    )
                else:
                    model = SARIMAX(y_train, order=(1, 1, 1), enforce_stationarity=False, enforce_invertibility=False)
                results = model.fit(method='powell', maxiter=1000, disp=False)
                search_results.append((with_mw, list(subset), results.aic))
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_subset = full_subset
                    print(f"\u2705 New best AIC: {best_aic:.2f} with vars: {best_subset}")
            except Exception as e:
                print(f"\u26A0\uFE0F Skipped {subset}: {e}")
            count += 1
            if count % 200 == 0:
                print(count, "models fit so far...")

print("\n\U0001F3AF Search complete.")
print("Best AIC:", best_aic)
print("Best subset of variables:", best_subset)

In [ ]:
aics_no_mw = [aic for with_mw, subset, aic in search_results if not with_mw]
aics_mw = [aic for with_mw, subset, aic in search_results if with_mw]
best = min(aics_no_mw + aics_mw)

d_no = [a - best for a in aics_no_mw]
d_mw = [a - best for a in aics_mw]

plt.figure(figsize=(10, 5))
bins = np.linspace(0, np.percentile(d_no + d_mw, 99), 45)
plt.hist(d_no, bins=bins, color='#c0392b', alpha=0.55, label='Models without minimum wage', edgecolor='white', linewidth=0.4)
plt.hist(d_mw, bins=bins, color='#117733', alpha=0.55, label='Models with minimum wage', edgecolor='white', linewidth=0.4)
plt.axvline(2, color='#1f3b6e', ls='--', lw=1.6, label='\u0394AIC = 2 (indistinguishability threshold)')
plt.xlabel('\u0394AIC relative to best-fitting model')
plt.ylabel('Number of model combinations')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('fig3_aic_hist.png', dpi=200, bbox_inches='tight')
plt.show()

# how often does adding min wage actually help?
pairs = {}
for with_mw, subset, aic in search_results:
    key = tuple(subset)
    pairs.setdefault(key, {})[with_mw] = aic
deltas = [v[True] - v[False] for v in pairs.values() if True in v and False in v]
print("min wage lowers AIC in", sum(1 for d in deltas if d < 0), "of", len(deltas), "matched pairs")
print("share of pairs with |delta| < 2:", round(100 * np.mean([abs(d) < 2 for d in deltas]), 1), "%")

In [ ]:
# min / max / mean / std of every variable before standardization (for Table 3)
full_data.describe().T[['mean', 'std', 'min', 'max']].round(2)